In [1]:
from pathlib import Path
from tqdm.notebook import tqdm
import time

import torch
from torch.utils.data import random_split
from torch_geometric.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from halide_gnn_cost_model.data import PipelineDataset
from halide_gnn_cost_model.data import graph_transformer_preprocessor
from halide_gnn_cost_model.model import PipeGPS

In [2]:
PIPELINES_DIR = Path("/lscratch/fzhou48/pipelines-16k-data")
GRAPH_TRANSFORMER_MODEL_DIR = Path("/home/groups/kayvonf/fzhou48/halide-gnn-cost-model/resources/models/graph_transformer")
GRAPH_TRANSFORMER_MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
USE_GPU = True

if USE_GPU and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_GPU and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

Using device: cuda


In [4]:
# Load dataset
dataset = PipelineDataset(PIPELINES_DIR, preprocessor=graph_transformer_preprocessor)

1lines [00:00, 4156.89lines/s]
1lines [00:00, 14614.30lines/s]
1lines [00:00, 14926.35lines/s]
1lines [00:00, 15252.01lines/s]


In [5]:
# Train/test split
num_train = int(0.95 * len(dataset))
train_dataset, test_dataset = random_split(dataset, [num_train, len(dataset) - num_train])
len(train_dataset), len(test_dataset)

(15605, 822)

# Model

In [6]:
vocab_size = len(dataset.ast_vocab) + len(dataset.sched_vocab) + 1
model = PipeGPS(
    hidden_channels=64,
    num_layers=8,
    num_attn_heads=4,
    attn_type="multihead",
    attn_kwargs={"dropout": 0.5},
    vocab_size=vocab_size,
    num_runtime_targets=5,
)
model = model.to(device)

# Train model

In [7]:
data_loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)
data_loader

/tmp/ipykernel_194106/958583031.py:1: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  data_loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)


In [8]:
# Save model every n epochs
SAVE_EVERY = 10
NUM_EPOCHS = 100

summary_writer = SummaryWriter(log_dir="resources/runs/" + time.strftime("graph-transformer-%Y%m%d-%H%M%S"))
# Train loop (example)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = torch.nn.MSELoss()

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    total_loss = 0
    for batch_idx, data in enumerate(data_loader):
        optimizer.zero_grad()
        data = data.to(device)
        x = data.type
        edge_index = data.edge_index
        pe = torch.cat((data.laplacian_eigenvector_pe, data.random_walk_pe, data.degree_pe), dim=-1)
        node_type = data.node_type
        pred = model(x, pe, node_type, edge_index, data.batch)
        # Assuming batch.y contains the true runtimes
        loss = criterion(pred.reshape(-1), torch.log(data.y))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        summary_writer.add_scalar("Loss/train", loss.item(), epoch * len(data_loader) + batch_idx)
    # Print average loss for the epoch
    avg_loss = total_loss / len(data_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    # Save model
    if (epoch + 1) % SAVE_EVERY == 0:
        torch.save(model.state_dict(), GRAPH_TRANSFORMER_MODEL_DIR / f"pipeline_model_epoch_{epoch+1}.pt")

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1, Loss: 28.1024
Epoch 2, Loss: 6.0758
Epoch 3, Loss: 4.7874
Epoch 4, Loss: 4.1290
Epoch 5, Loss: 3.6841
Epoch 6, Loss: 3.1135
Epoch 7, Loss: 2.8228
Epoch 8, Loss: 2.5598
